# Import thư viện, Khởi tạo RNN và Metric

In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, matthews_corrcoef,
    cohen_kappa_score, confusion_matrix
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
class TimeSeriesRNNDataset(Dataset):
    def __init__(self, X, y):
        # X: (số mẫu, 4 phase, số feature mỗi phase)
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        # Trả về 1 chuỗi 4 phase + nhãn tương ứng
        return self.X[idx], self.y[idx]

class TimeSeriesRNN(nn.Module):
    def __init__(
        self,
        input_dim_per_phase,   # số feature mỗi phase
        hidden_dim=64,
        num_layers=2,
        num_classes=3,
        dropout=0.3,
        bidirectional=False
    ):
        super().__init__()

        # RNN xử lý chuỗi 4 phase
        self.rnn = nn.RNN(
            input_size=input_dim_per_phase,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional,
            nonlinearity='tanh'   # mặc định của RNN
        )

        # Nếu BiRNN thì hidden_dim * 2
        multiplier = 2 if bidirectional else 1

        # Fully connected để phân loại
        self.fc = nn.Linear(hidden_dim * multiplier, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, 4, features)
        out, hn = self.rnn(x)

        # Lấy output của phase cuối (phase 4)
        out = out[:, -1, :]
        out = self.dropout(out)

        return self.fc(out)


In [3]:
# Metrics
def gmean_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    per_class = []
    for i in range(cm.shape[0]):
        tp = cm[i,i]
        fn = cm[i].sum() - tp
        fp = cm[:,i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        per_class.append(np.sqrt(sens * spec))
    return np.prod(per_class) ** (1/len(per_class)) if per_class else 0

def gmean_per_class(y_true, y_pred, target_class):
    cm = confusion_matrix(y_true, y_pred)
    i = target_class
    tp = cm[i,i]
    fn = cm[i].sum() - tp
    fp = cm[:,i].sum() - tp
    tn = cm.sum() - tp - fn - fp

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    return np.sqrt(recall * specificity)

def print_results(version_name, phase, y_true, y_pred):
    target_names = ['Excellent', 'Good', 'Average']
    print(f"\n{'='*30} {version_name} - Phase {phase} {'='*30}")
    print(classification_report(y_true, y_pred, digits=10, target_names=target_names))

    # G-Mean per class
    print("G-Mean per class (one-vs-rest):")
    for idx, name in enumerate(target_names):
        g = gmean_per_class(y_true, y_pred, idx)
        print(f"  {name:<10}: {g:.10f}")
        
    print()
    
    metrics = {
        'Version': version_name,
        'Phase': phase,
        'Accuracy': accuracy_score(y_true, y_pred),
        'BalancedAcc': balanced_accuracy_score(y_true, y_pred),
        'PrecMacro': precision_score(y_true, y_pred, average='macro'),
        'PrecWeighted': precision_score(y_true, y_pred, average='weighted'),
        'RecMacro': recall_score(y_true, y_pred, average='macro'),
        'RecWeighted': recall_score(y_true, y_pred, average='weighted'),
        'F1Macro': f1_score(y_true, y_pred, average='macro'),
        'F1Weighted': f1_score(y_true, y_pred, average='weighted'),
        'GMean': gmean_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'Kappa': cohen_kappa_score(y_true, y_pred)
    }
    
    for k, v in metrics.items():
        if k not in ['Version', 'Phase']:
            print(f"{k:18} : {v:.10f}")
    
    return metrics

# Hàm chuẩn bị dữ liệu và train

In [4]:
def prepare_and_train_rnn(train_path, val_path=None, test_paths=None):
    print(f"Loading train: {train_path}")
    df_train = pd.read_csv(train_path)
    
    # Nếu có validation thì gộp train + val để train chung
    if val_path:
        df_val = pd.read_csv(val_path)
        df = pd.concat([df_train, df_val], ignore_index=True)
        print(f"Combined train + val: {len(df)} samples")
    else:
        df = df_train
        print(f"Only train: {len(df)} samples")
    
    df = df.drop(columns=['user_id', 'course_id'], errors='ignore')
    
    # Tách nhãn
    y = df['label_3'].values
    df_features = df.drop('label_3', axis=1)
    
    # Lấy các cột thuộc 4 phase (_1, _2, _3, _4)
    phase_cols = [col for col in df_features.columns if col.endswith(('_1', '_2', '_3', '_4'))]
    
    # Tên feature gốc (không bao gồm hậu tố phase)
    base_names = sorted(set(col.rsplit('_', 1)[0] for col in phase_cols))
    
    # Tạo dữ liệu chuỗi thời gian (4 phase)
    phases_data = []
    for p in ['1', '2', '3', '4']:
        cols_p = [col for col in phase_cols if col.endswith(f'_{p}')]
        phases_data.append(df_features[cols_p].values)
    
    # Shape: (samples, 4 phase, số feature mỗi phase)
    X = np.stack(phases_data, axis=1)
    
    print(f"X shape: {X.shape} → (samples, seq_len=4, features_per_phase)")
    
    # Chuẩn hóa dữ liệu
    scaler = StandardScaler()
    N, T, F = X.shape
    
    # Scale theo feature
    X_reshaped = X.reshape(-1, F)
    X_scaled = scaler.fit_transform(X_reshaped)
    X = X_scaled.reshape(N, T, F)

    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    print(f"Classes: {le.classes_} → {le.transform(le.classes_)}")

    dataset = TimeSeriesRNNDataset(X, y_enc)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    
    model = TimeSeriesRNN(
        input_dim_per_phase=F,
        hidden_dim=64,
        num_layers=2,
        dropout=0.3
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Huấn luyện mô hình (có early stopping)
    model.train()
    best_loss = float('inf')
    patience = 10
    wait = 0
    
    for epoch in range(80):
        epoch_loss = 0
        
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        epoch_loss /= len(loader)
        
        # Lưu model tốt nhất
        if epoch_loss < best_loss - 1e-4:
            best_loss = epoch_loss
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    model.load_state_dict(best_state)
    model.eval()
    print("Training done")
    
    return model, scaler, le, base_names

# Chạy từng version

In [5]:
base_path = "/kaggle/input/mooccubex-data-cleaned/split_data"

In [6]:
def run_experiment(
    base_path,
    data_folder,
    train_file,
    val_file,
    test_prefix,
    version_name
):
    print(f"\n{'#'*5}")
    print(f"Version: {version_name}")
    print(f"{'#'*5}")

    train_path = f"{base_path}/{data_folder}/{train_file}"
    val_path   = f"{base_path}/{data_folder}/{val_file}"

    test_files = [
        f"{base_path}/{data_folder}/{test_prefix}_1.csv",
        f"{base_path}/{data_folder}/{test_prefix}_2.csv",
        f"{base_path}/{data_folder}/{test_prefix}_3.csv",
        f"{base_path}/{data_folder}/{test_prefix}_4.csv",
    ]

    # Huấn luyện mô hình RNN
    model, scaler, le, _ = prepare_and_train_rnn(train_path, val_path)
    results = []
    model.eval()
    with torch.no_grad():

        # Đánh giá mô hình trên từng phase test
        for phase, test_path in enumerate(test_files, 1):
            print(f"\n--- Test Phase {phase}: {test_path} ---")

            df_test = pd.read_csv(test_path)

            df_test = df_test.drop(columns=['user_id', 'course_id'], errors='ignore')

            # Tách nhãn
            y_test = df_test['label_3'].values
            df_test = df_test.drop('label_3', axis=1)

            # Tách đặc trưng theo 4 phase
            phase_cols = [
                col for col in df_test.columns
                if col.endswith(('_1', '_2', '_3', '_4'))
            ]

            phases_data = []
            for p in ['1', '2', '3', '4']:
                cols_p = [c for c in phase_cols if c.endswith(f'_{p}')]
                phases_data.append(df_test[cols_p].values)

            # Dạng dữ liệu cuối: (số mẫu, seq_len=4, số đặc trưng mỗi phase)
            X_test = np.stack(phases_data, axis=1)

            # Chuẩn hóa dữ liệu (Dùng scaler đã fit từ tập train)
            N, T, F = X_test.shape
            X_test_scaled = scaler.transform(
                X_test.reshape(-1, F)
            ).reshape(N, T, F)

            # Dự đoán bằng mô hình GRU
            X_tensor = torch.FloatTensor(X_test_scaled).to(device)
            preds = torch.argmax(model(X_tensor), dim=1).cpu().numpy()

            # Encode nhãn
            y_test_enc = le.transform(y_test)

            res = print_results(version_name, phase, y_test_enc, preds)
            results.append(res)

    return pd.DataFrame(results).round(10)

## V0 (Raw Median)

In [7]:
df_v0 = run_experiment(
    base_path=base_path,
    data_folder="data_median_im_3",
    train_file="train_us.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V0 (Raw Median)"
)

df_v0


#####
Version: V0 (Raw Median)
#####
Loading train: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/train_us.csv
Combined train + val: 439074 samples
X shape: (439074, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/test_1.csv ---

============================== V0 (Raw Median) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.0000000000 0.0000000000 0.0000000000       223
        Good  0.0000000000 0.0000000000 0.0000000000       605
     Average  0.9964379896 1.0000000000 0.9982158172    231625

    accuracy                      0.9964379896    232453
   macro avg  0.3321459965 0.3333333333 0.3327386057    232453
weighted avg  0.9928886671 0.9964379896 0.9946601621    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.0000000000
  Good      : 0.0000000000
  Average   : 0.0000

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V0 (Raw Median),1,0.996438,0.333333,0.332146,0.992889,0.333333,0.996438,0.332739,0.994660,0.000000,0.000000,0.000000
1,V0 (Raw Median),2,0.996438,0.333333,0.332146,0.992889,0.333333,0.996438,0.332739,0.994660,0.000000,0.000000,0.000000
2,V0 (Raw Median),3,0.996305,0.338784,0.396264,0.993436,0.338784,0.996305,0.342864,0.994675,0.000000,0.049907,0.023508
3,V0 (Raw Median),4,0.996705,0.823715,0.710157,0.997768,0.823715,0.996705,0.752088,0.997120,0.879927,0.650128,0.634976


## V1 (Median CDS)

In [8]:
df_v1 = run_experiment(
    base_path=base_path,
    data_folder="data_median_cdsmote",
    train_file="train_median_cdsmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V1 (Median CDS)"
)

df_v1


#####
Version: V1 (Median CDS)
#####
Loading train: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_cdsmote/train_median_cdsmote.csv
Combined train + val: 832452 samples
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_cdsmote/test_1.csv ---

============================== V1 (Median CDS) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.0000000000 0.0000000000 0.0000000000       223
        Good  0.0026300639 0.9966942149 0.0052462839       605
     Average  0.9990569003 0.0137204533 0.0270691550    231625

    accuracy                      0.0162656537    232453
   macro avg  0.3338956547 0.3368048894 0.0107718129    232453
weighted avg  0.9955050945 0.0162656537 0.0269863887    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.0000000000
  Good      : 0.1169027038
  

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V1 (Median CDS),1,0.016266,0.336805,0.333896,0.995505,0.336805,0.016266,0.010772,0.026986,0.000000,0.004539,0.000064
1,V1 (Median CDS),2,0.808047,0.500013,0.336029,0.995225,0.500013,0.808047,0.304137,0.890863,0.000000,0.067415,0.016244
2,V1 (Median CDS),3,0.996533,0.439436,0.610224,0.995537,0.439436,0.996533,0.466212,0.995845,0.378698,0.377434,0.357962
3,V1 (Median CDS),4,0.996971,0.528716,0.742843,0.996221,0.528716,0.996971,0.597605,0.996411,0.558537,0.455718,0.427954


## V2 (Median SAS)

In [9]:
df_v2 = run_experiment(
    base_path=base_path,
    data_folder="data_median_sasmote",
    train_file="train_median_sasmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V2 (Median SAS)"
)

df_v2


#####
Version: V2 (Median SAS)
#####
Loading train: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_sasmote/train_median_sasmote.csv
Combined train + val: 832452 samples
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_sasmote/test_1.csv ---

============================== V2 (Median SAS) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.0000000000 0.0000000000 0.0000000000       223
        Good  0.2009456265 0.4214876033 0.2721451441       605
     Average  0.9980491223 0.9961230437 0.9970851529    231625

    accuracy                      0.9936718390    232453
   macro avg  0.3996649163 0.4725368823 0.4230767656    232453
weighted avg  0.9950170575 0.9936718390 0.9942418310    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.0000000000
  Good      : 0.6477995091
  

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V2 (Median SAS),1,0.993672,0.472537,0.399665,0.995017,0.472537,0.993672,0.423077,0.994242,0.000000,0.305000,0.297902
1,V2 (Median SAS),2,0.007873,0.363164,0.395882,0.984727,0.363164,0.007873,0.103039,0.012473,0.167257,0.070405,0.000976
2,V2 (Median SAS),3,0.009490,0.479803,0.416920,0.994724,0.479803,0.009490,0.120582,0.015226,0.188244,0.098200,0.001333
3,V2 (Median SAS),4,0.993091,0.874676,0.551262,0.997350,0.874676,0.993091,0.642123,0.994791,0.924045,0.535384,0.474644


## V3 (Median Radius)

In [10]:
df_v3 = run_experiment(
    base_path=base_path,
    data_folder="data_median_radiussmote",
    train_file="train_median_radiussmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V3 (Median Radius)"
)

df_v3


#####
Version: V3 (Median Radius)
#####
Loading train: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_radiussmote/train_median_radiussmote.csv
Combined train + val: 832452 samples
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_radiussmote/test_1.csv ---

============================== V3 (Median Radius) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.4716981132 0.1121076233 0.1811594203       223
        Good  0.0025387263 0.9752066116 0.0050642690       605
     Average  0.0000000000 0.0000000000 0.0000000000    231625

    accuracy                      0.0026456961    232453
   macro avg  0.1580789465 0.3624380783 0.0620745631    232453
weighted avg  0.0004591234 0.0026456961 0.0001869730    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.3348045796
  Good     

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V3 (Median Radius),1,0.002646,0.362438,0.158079,0.000459,0.362438,0.002646,0.062075,0.000187,0.000000,0.024113,0.000044
1,V3 (Median Radius),2,0.002538,0.347015,0.394967,0.996619,0.347015,0.002538,0.047914,0.000172,0.028744,-0.026121,-0.000077
2,V3 (Median Radius),3,0.007016,0.507216,0.381220,0.993688,0.507216,0.007016,0.086321,0.009069,0.168667,0.000533,0.000006
3,V3 (Median Radius),4,0.992046,0.837400,0.505838,0.997160,0.837400,0.992046,0.589525,0.994142,0.900758,0.497059,0.431780


## V4 (Tree CDS)

In [11]:
df_v4 = run_experiment(
    base_path=base_path,
    data_folder="data_extra_trees_cdsmote",
    train_file="train_extratrees_cdsmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V4 (Tree CDS)"
)

df_v4


#####
Version: V4 (Tree CDS)
#####
Loading train: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_cdsmote/train_extratrees_cdsmote.csv
Combined train + val: 832452 samples
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_cdsmote/test_1.csv ---

============================== V4 (Tree CDS) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.0000000000 0.0000000000 0.0000000000       223
        Good  0.0000000000 0.0000000000 0.0000000000       605
     Average  0.9964379896 1.0000000000 0.9982158172    231625

    accuracy                      0.9964379896    232453
   macro avg  0.3321459965 0.3333333333 0.3327386057    232453
weighted avg  0.9928886671 0.9964379896 0.9946601621    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.0000000000
  Good      : 0.000

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V4 (Tree CDS),1,0.996438,0.333333,0.332146,0.992889,0.333333,0.996438,0.332739,0.994660,0.000000,0.000000,0.000000
1,V4 (Tree CDS),2,0.996442,0.334434,0.465485,0.993947,0.334434,0.996442,0.334927,0.994684,0.000000,0.046453,0.007166
2,V4 (Tree CDS),3,0.990359,0.621501,0.434171,0.995778,0.621501,0.990359,0.481584,0.992802,0.695266,0.331179,0.290553
3,V4 (Tree CDS),4,0.990032,0.896662,0.487776,0.997131,0.896662,0.990032,0.574446,0.992932,0.937014,0.475637,0.391054


## V5 (Tree SAS)

In [12]:
df_v5 = run_experiment(
    base_path=base_path,
    data_folder="data_extra_trees_sasmote",
    train_file="train_extratrees_sasmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V5 (Tree SAS)"
)

df_v5


#####
Version: V5 (Tree SAS)
#####
Loading train: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_sasmote/train_extratrees_sasmote.csv
Combined train + val: 832452 samples
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_sasmote/test_1.csv ---

============================== V5 (Tree SAS) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.0000000000 0.0000000000 0.0000000000       223
        Good  0.0000000000 0.0000000000 0.0000000000       605
     Average  0.9964422150 0.9999827307 0.9982093334    231625

    accuracy                      0.9964207818    232453
   macro avg  0.3321474050 0.3333275769 0.3327364445    232453
weighted avg  0.9928928775 0.9964207818 0.9946537014    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.0000000000
  Good      : 0.000

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V5 (Tree SAS),1,0.996421,0.333328,0.332147,0.992893,0.333328,0.996421,0.332736,0.994654,0.000000,0.007606,0.001173
1,V5 (Tree SAS),2,0.984027,0.462659,0.439294,0.994818,0.462659,0.984027,0.407899,0.989226,0.477452,0.122128,0.097383
2,V5 (Tree SAS),3,0.989972,0.650555,0.442623,0.996212,0.650555,0.989972,0.493043,0.992779,0.717585,0.364834,0.312561
3,V5 (Tree SAS),4,0.991323,0.901418,0.501609,0.997231,0.901418,0.991323,0.591210,0.993703,0.939501,0.501507,0.424671


## V6 (Tree Radius)

In [13]:
df_v6 = run_experiment(
    base_path=base_path,
    data_folder="data_extra_trees_radiussmote",
    train_file="train_extratrees_radiussmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V6 (Tree Radius)"
)

df_v6


#####
Version: V6 (Tree Radius)
#####
Loading train: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_radiussmote/train_extratrees_radiussmote.csv
Combined train + val: 832452 samples
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_radiussmote/test_1.csv ---

============================== V6 (Tree Radius) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.0158730159 0.0044843049 0.0069930070       223
        Good  0.0028867859 0.6595041322 0.0057484098       605
     Average  0.9970798734 0.4053923368 0.5764228865    231625

    accuracy                      0.4056691030    232453
   macro avg  0.3386132251 0.3564602580 0.1963881011    232453
weighted avg  0.9935510055 0.4056691030 0.5743913321    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.0669560134
 

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V6 (Tree Radius),1,0.405669,0.356460,0.338613,0.993551,0.356460,0.405669,0.196388,0.574391,0.262164,0.007398,0.000728
1,V6 (Tree Radius),2,0.943726,0.521989,0.348021,0.994861,0.521989,0.943726,0.351745,0.968018,0.573510,0.109678,0.050886
2,V6 (Tree Radius),3,0.993969,0.615362,0.498090,0.995648,0.615362,0.993969,0.538511,0.994738,0.669419,0.363085,0.352584
3,V6 (Tree Radius),4,0.990725,0.855415,0.485936,0.997063,0.855415,0.990725,0.568126,0.993340,0.909934,0.472822,0.397484
